In [ ]:
# Load R magic extension for Python Jupyter kernel (Kaggle / Colab support)
try:
    %load_ext rpy2.ipython
except Exception as e:
    print("Note on rpy2 initialization:", e)

# Dual Specialist LightGBM Classifier: LightGBM for ESI 2/3 & LightGBM for ESI 4/5 (`models/xgboost_esi23_esi45_extreme.ipynb`)

This notebook trains a **Dual Specialist Ensemble Model** using **LightGBM for both sub-models** (LightGBM Specialist 1 for ESI 2/3 and LightGBM Specialist 2 for ESI 4/5) with **35 Predictor Features** (19 raw inputs from `config/triage_conf.json` + 16 continuous vital delta & range features; strictly excluding binary vital anomaly flags), **Factor-Controlled Minority Class Random Upsampling**, evaluated strictly on **Recall**, **Specificity**, **Balanced Accuracy**, and **ROC-AUC**:

### System Architecture & Workflow
1. **Specialist Model 1 (LightGBM for ESI 2/3)**:
   - Target: Binary `ESI 2/3` (`1` if ESI 2 or 3, `0` otherwise).
   - Algorithm: LightGBM (`objective = "binary"`, `metric = "binary_logloss"`).
   - Predicts $P_{\text{LGB1}}(\text{ESI 2/3})$.
2. **Specialist Model 2 (LightGBM for ESI 4/5)**:
   - Target: Binary `ESI 4/5` (`1` if ESI 4 or 5, `0` otherwise).
   - Algorithm: LightGBM (`objective = "binary"`, `metric = "binary_logloss"`).
   - Predicts $P_{\text{LGB2}}(\text{ESI 4/5})$.
3. **Predictor Feature Selection (35 Total Features)**:
   - **19 Raw Inputs (From `triage_conf.json`)**: `age`, `gender`, `cc_breathingdifficulty`, `triage_vital_hr`, `triage_vital_sbp`, `triage_vital_rr`, `triage_vital_o2`, `pulse_last`, `resp_last`, `spo2_last`, `sbp_last`, `pulse_min`, `resp_min`, `spo2_min`, `sbp_min`, `pulse_max`, `resp_max`, `spo2_max`, `sbp_max`.
   - **16 Continuous Vital Delta & Range Features**: `hr_mean_to_last`, `sbp_mean_to_last`, `spo2_mean_to_last`, `rr_mean_to_last`, `hr_range`, `rr_range`, `spo2_range`, `sbp_range`, `hr_last_to_min`, `rr_last_to_min`, `spo2_last_to_min`, `sbp_last_to_min`, `hr_last_to_max`, `rr_last_to_max`, `spo2_last_to_max`, `sbp_last_to_max`.
4. **Single Validation Split Partitioning (From Config)**:
   - Stratified Partitioning into Train, Validation (`val_size`), and Test (`test_size`) sets parsed directly from `config/triage_conf.json`.
   - Applies **Minority Class Bootstrap Random Upsampling** (`upsample_ratio = 1.0`) strictly to training partitions.
5. **Targeted Benchmarking Suite**: Evaluates ONLY **Recall (Sensitivity)**, **Specificity**, **Balanced Accuracy**, and **ROC-AUC**.
6. **Reports & Artifacts**:
   - Saved separately to `deploy/lightgbm_esi23_model.rds`, `deploy/lightgbm_esi45_model.rds`, and `deploy/xgboost_esi23_esi45_extreme_model.rds`.

In [ ]:
%%R
# ---------------------------------------------------------
# Step 1: Load Required Libraries & Parse Configuration JSON
# ---------------------------------------------------------
suppressPackageStartupMessages({
  library(jsonlite)
  library(caret)
  library(dplyr)
  library(ggplot2)
  library(tidyr)
  library(pROC)
  library(xgboost)
})
has_lgb <- requireNamespace("lightgbm", quietly = TRUE)
if (has_lgb) {
  library(lightgbm)
  cat("LightGBM R package successfully loaded for both Dual Specialist Models.\n")
} else {
  cat("Note: LightGBM R package not installed. Falling back to XGBoost binary gradient boosting for Dual Specialist Models.\n")
}
config_path <- "../config/triage_conf.json"
if (!file.exists(config_path)) {
  config_path <- "config/triage_conf.json"
}
config <- fromJSON(config_path)
cat("=== Configuration Loaded from config/triage_conf.json ===\n")
cat("Data Source Path:", config$path$data_source, "\n")
cat("Target Column:   ", config$classes$target_col, "\n")
cat("Test Size:       ", config$training$test_size, "\n")
cat("Val Size:        ", config$training$val_size, "\n")
cat("Random State:    ", config$training$random_state, "\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 2: Load Data, Construct 35 Predictor Features & Define Binary Target Labels
# ---------------------------------------------------------
set.seed(config$training$random_state)
data_file <- config$path$data_source
if (!file.exists(data_file) && file.exists(paste0("../", data_file))) {
  data_file <- paste0("../", data_file)
}
cat("Loading dataset from:", data_file, "...\n")
data_env <- new.env()
load(data_file, envir = data_env)
df_names <- ls(data_env)[sapply(ls(data_env), function(x) is.data.frame(get(x, envir = data_env)))]
df_sizes <- sapply(df_names, function(x) nrow(get(x, envir = data_env)))
data_obj_name <- df_names[which.max(df_sizes)]
cat(sprintf("Selected main dataset object: '%s' (%d rows)\n", data_obj_name, max(df_sizes)))
raw_df <- get(data_obj_name, envir = data_env)
target_col_name <- config$classes$target_col
gender_vec <- if ("gender" %in% names(raw_df)) ifelse(as.character(raw_df$gender) == "Male", 1, 0) else 0
cc_bd_vec  <- if ("cc_breathingdifficulty" %in% names(raw_df)) ifelse(!is.na(raw_df$cc_breathingdifficulty), raw_df$cc_breathingdifficulty, 0) else 0
get_vec <- function(col_name, default_val = 0) {
  if (col_name %in% names(raw_df)) {
    res <- raw_df[[col_name]]
    res[is.na(res)] <- default_val
    return(res)
  } else {
    return(rep(default_val, nrow(raw_df)))
  }
}
p_last   <- get_vec("pulse_last")
p_max    <- get_vec("pulse_max")
p_min    <- get_vec("pulse_min")
s_last   <- get_vec("sbp_last")
s_max    <- get_vec("sbp_max")
s_min    <- get_vec("sbp_min")
o2_last  <- get_vec("spo2_last")
o2_max   <- get_vec("spo2_max")
o2_min   <- get_vec("spo2_min")
r_last   <- get_vec("resp_last")
r_max    <- get_vec("resp_max")
r_min    <- get_vec("resp_min")
t_hr     <- get_vec("triage_vital_hr")
t_sbp    <- get_vec("triage_vital_sbp")
t_o2     <- get_vec("triage_vital_o2")
t_rr     <- get_vec("triage_vital_rr")
# Construct 35 Predictor Features (19 raw triage + 16 continuous vital deltas & ranges)
df_full <- data.frame(
  # 19 Raw Inputs from triage_conf.json
  age                     = raw_df$age,
  gender                  = gender_vec,
  cc_breathingdifficulty  = cc_bd_vec,
  triage_vital_hr         = t_hr,
  triage_vital_sbp        = t_sbp,
  triage_vital_rr         = t_rr,
  triage_vital_o2         = t_o2,
  pulse_last              = p_last,
  resp_last               = r_last,
  spo2_last               = o2_last,
  sbp_last                = s_last,
  pulse_min               = p_min,
  resp_min                = r_min,
  spo2_min                = o2_min,
  sbp_min                 = s_min,
  pulse_max               = p_max,
  resp_max                = r_max,
  spo2_max                = o2_max,
  sbp_max                 = s_max,
  
  # 16 Continuous Vital Delta & Range Features
  hr_mean_to_last         = t_hr - p_last,
  sbp_mean_to_last        = t_sbp - s_last,
  spo2_mean_to_last       = t_o2 - o2_last,
  rr_mean_to_last         = t_rr - r_last,
  
  hr_range                = p_max - p_min,
  rr_range                = r_max - r_min,
  spo2_range              = o2_max - o2_min,
  sbp_range               = s_max - s_min,
  
  hr_last_to_min          = p_last - p_min,
  rr_last_to_min          = r_last - r_min,
  spo2_last_to_min        = o2_last - o2_min,
  sbp_last_to_min         = s_last - s_min,
  
  hr_last_to_max          = p_last - p_max,
  rr_last_to_max          = r_last - r_max,
  spo2_last_to_max        = o2_last - o2_max,
  sbp_last_to_max         = s_last - s_max
)
raw_esi <- as.character(raw_df[[target_col_name]])
df_full$target_col   <- factor(ifelse(raw_esi %in% c("2", "3"), "2_3", ifelse(raw_esi %in% c("4", "5"), "4_5", "1")), levels = c("2_3", "4_5", "1"))
df_full$target_esi23 <- ifelse(raw_esi %in% c("2", "3"), 1, 0)
df_full$target_esi45 <- ifelse(raw_esi %in% c("4", "5"), 1, 0)
initial_rows <- nrow(df_full)
df_full <- na.omit(df_full)
cat(sprintf("Complete Case Filtering: Removed %d rows (Remaining: %d)\n", initial_rows - nrow(df_full), nrow(df_full)))
cat(sprintf("Full Dataset Ready (35 Features): %d total rows x %d cols\n", nrow(df_full), ncol(df_full)))
print(table(df_full$target_col))

In [ ]:
%%R
# ---------------------------------------------------------
# Step 3: Single Validation Split Partitioning (Strictly from config/triage_conf.json)
# ---------------------------------------------------------
set.seed(config$training$random_state)
test_size <- config$training$test_size
val_size  <- config$training$val_size
in_train_val <- createDataPartition(df_full$target_col, p = 1 - test_size, list = FALSE)
train_val_df <- df_full[in_train_val, ]
test_df      <- df_full[-in_train_val, ]
rel_val_size <- val_size / (1 - test_size)
in_train    <- createDataPartition(train_val_df$target_col, p = 1 - rel_val_size, list = FALSE)
train_df    <- train_val_df[in_train, ]
val_df      <- train_val_df[-in_train, ]
upsample_ratio <- 1.0
upsample_multiclass <- function(df_train, ratio = 1.0) {
  counts <- table(df_train$target_col)
  max_cnt <- max(counts)
  
  res_df <- df_train
  for (cls in names(counts)) {
    cls_cnt <- counts[[cls]]
    target_cnt <- round(max_cnt * ratio)
    if (target_cnt > cls_cnt) {
      extra_needed <- target_cnt - cls_cnt
      cls_subset   <- df_train[df_train$target_col == cls, ]
      sampled_extra <- cls_subset[sample(1:cls_cnt, size = extra_needed, replace = TRUE), ]
      res_df       <- rbind(res_df, sampled_extra)
    }
  }
  return(res_df)
}
train_df_upsampled <- upsample_multiclass(train_df, ratio = upsample_ratio)
binary_cols <- c("gender", "cc_breathingdifficulty")
feature_cols <- setdiff(names(train_df), c(binary_cols, "target_col", "target_esi23", "target_esi45"))
cont_cols    <- feature_cols
preproc <- preProcess(train_df_upsampled[, cont_cols, drop = FALSE], method = c("center", "scale"))
train_scaled <- predict(preproc, train_df_upsampled)
val_scaled   <- predict(preproc, val_df)
test_scaled  <- predict(preproc, test_df)
all_feats <- c(binary_cols, cont_cols)
train_x <- as.matrix(train_scaled[, all_feats])
val_x   <- as.matrix(val_scaled[, all_feats])
test_x  <- as.matrix(test_scaled[, all_feats])
cat(sprintf("Dataset Splits (from config): Train = %d, Val = %d, Test = %d\n",
            nrow(train_scaled), nrow(val_scaled), nrow(test_scaled)))

In [ ]:
%%R
# ---------------------------------------------------------
# Step 4: Train Dual Specialist LightGBM Models & Save Separate Artifacts
# ---------------------------------------------------------
# 1. Train LightGBM Model for ESI 2/3 Specialist
y_tr_23  <- train_scaled$target_esi23
y_val_23 <- val_scaled$target_esi23
if (has_lgb) {
  dtr_lgb23 <- lgb.Dataset(data = train_x, label = y_tr_23)
  dvl_lgb23 <- lgb.Dataset(data = val_x,   label = y_val_23, reference = dtr_lgb23)
  lgb_params23 <- list(objective = "binary", metric = "binary_logloss", learning_rate = 0.05, num_leaves = 31, max_depth = 6, verbosity = -1)
  final_lgb_23 <- lgb.train(params = lgb_params23, data = dtr_lgb23, nrounds = 150, valids = list(val = dvl_lgb23), early_stopping_rounds = 20, verbose = -1)
} else {
  dtr_xgb23 <- xgb.DMatrix(data = train_x, label = y_tr_23)
  dvl_xgb23 <- xgb.DMatrix(data = val_x,   label = y_val_23)
  final_lgb_23 <- xgb.train(params = list(objective = "binary:logistic", eval_metric = "logloss", eta = 0.05, max_depth = 6), data = dtr_xgb23, nrounds = 150, evals = list(val = dvl_xgb23), early_stopping_rounds = 20, verbose = 0)
}
# 2. Train LightGBM Model for ESI 4/5 Specialist
y_tr_45  <- train_scaled$target_esi45
y_val_45 <- val_scaled$target_esi45
if (has_lgb) {
  dtr_lgb45 <- lgb.Dataset(data = train_x, label = y_tr_45)
  dvl_lgb45 <- lgb.Dataset(data = val_x,   label = y_val_45, reference = dtr_lgb45)
  lgb_params45 <- list(objective = "binary", metric = "binary_logloss", learning_rate = 0.05, num_leaves = 31, max_depth = 6, verbosity = -1)
  final_lgb_45 <- lgb.train(params = lgb_params45, data = dtr_lgb45, nrounds = 150, valids = list(val = dvl_lgb45), early_stopping_rounds = 20, verbose = -1)
} else {
  dtr_xgb45 <- xgb.DMatrix(data = train_x, label = y_tr_45)
  dvl_xgb45 <- xgb.DMatrix(data = val_x,   label = y_val_45)
  final_lgb_45 <- xgb.train(params = list(objective = "binary:logistic", eval_metric = "logloss", eta = 0.05, max_depth = 6), data = dtr_xgb45, nrounds = 150, evals = list(val = dvl_xgb45), early_stopping_rounds = 20, verbose = 0)
}
deploy_dir <- "../deploy"
if (!dir.exists(deploy_dir)) deploy_dir <- "deploy"
if (!dir.exists(deploy_dir)) dir.create(deploy_dir, recursive = TRUE)
saveRDS(list(model = final_lgb_23, preproc = preproc, has_lgb = has_lgb), file = file.path(deploy_dir, "lightgbm_esi23_model.rds"))
saveRDS(list(model = final_lgb_45, preproc = preproc, has_lgb = has_lgb), file = file.path(deploy_dir, "lightgbm_esi45_model.rds"))
saveRDS(list(model_lgb23 = final_lgb_23, model_lgb45 = final_lgb_45, model_lgb = final_lgb_23, model_xgb = final_lgb_45, preproc = preproc, upsample_ratio = upsample_ratio, has_lgb = has_lgb), file = file.path(deploy_dir, "xgboost_esi23_esi45_extreme_model.rds"))
cat("Dual LightGBM Specialist Models (35 Features) saved to deploy/ directory!\n")